# Prediksi Risiko Stres Panas — Data Preparation

Bab ini menyiapkan data bebas kebocoran untuk Decision Tree, XGBoost, dan LSTM. Label merupakan kelas suhu terasa **hari berikutnya**; split kronologis menjaga validasi dan pengujian mencerminkan prediksi masa depan.

In [1]:
import json
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight, compute_sample_weight

DATA_PATH = Path("../dataset_cuaca_elnino_merged.csv")
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
df["time"] = pd.to_datetime(df["time"])
df = df.sort_values(["kota", "time"]).reset_index(drop=True)
print(f"Dataset awal: {df.shape[0]:,} × {df.shape[1]}")
print(f"Rentang: {df['time'].min().date()} s.d. {df['time'].max().date()}")

Dataset awal: 14,980 × 29
Rentang: 2021-01-01 s.d. 2026-11-10


## Bab 5.1 — Data cleaning

In [2]:
rows_with_nan = int(df.isna().any(axis=1).sum())
null_by_column = df.isna().sum()
print(f"Baris dengan minimal satu NaN sebelum cleaning: {rows_with_nan}")
display(null_by_column[null_by_column > 0].rename("Missing").to_frame())

df_clean = df.dropna().reset_index(drop=True)
assert len(df_clean) == 14952, f"Jumlah baris bersih tidak sesuai: {len(df_clean)}"
assert df_clean.isna().sum().sum() == 0
assert df_clean.duplicated().sum() == 0
rows_per_city = df_clean.groupby("kota").size()
assert rows_per_city.eq(2136).all(), rows_per_city.to_dict()
print(f"Baris setelah dropna: {len(df_clean):,}; sisa NaN: {df_clean.isna().sum().sum()}; duplikat: {df_clean.duplicated().sum()}")
display(rows_per_city.rename("Baris bersih per kota").to_frame())
print("Outlier meteorologi/ONI tidak dihapus karena merepresentasikan kejadian fisis yang valid.")

Baris dengan minimal satu NaN sebelum cleaning: 28


,Missing
temp_max_lag1d,7
apparent_temp_lag1d,7
precip_lag1d,7
temp_max_lag2d,14
apparent_temp_lag2d,14
precip_lag2d,14
temp_max_lag3d,21
apparent_temp_lag3d,21
precip_lag3d,21
target_temp_max_besok,7


Baris setelah dropna: 14,952; sisa NaN: 0; duplikat: 0


,Baris bersih per kota
kota,
Balikpapan,2136
Jakarta,2136
Jayapura,2136
Madiun,2136
Makasar,2136
Medan,2136
Surabaya,2136


Outlier meteorologi/ONI tidak dihapus karena merepresentasikan kejadian fisis yang valid.


## Bab 5.2 — Feature engineering dan feature selection

In [3]:
df_clean["heat_risk_besok"] = pd.cut(
    df_clean["target_apparent_temp_besok"],
    bins=[-np.inf, 33.0, 36.0, 39.0, np.inf],
    labels=[0, 1, 2, 3],
    right=False,
).astype("int64")

day_angle = 2 * np.pi * df_clean["day_of_year"] / 365.25
df_clean["sin_doy"] = np.sin(day_angle)
df_clean["cos_doy"] = np.cos(day_angle)
df_clean["temp_range_or_heat_gap"] = df_clean["apparent_temp_max"] - df_clean["temp_max"]
df_clean["apparent_temp_diff_1d"] = df_clean["apparent_temp_max"] - df_clean["apparent_temp_lag1d"]

leakage_columns = [
    "target_temp_max_besok", "target_apparent_temp_besok",
    "oni_anomaly", "oni_sst_total", "time", "day_of_year",
]
X_frame = df_clean.drop(columns=leakage_columns + ["heat_risk_besok"])
y = df_clean["heat_risk_besok"].astype("int64")
assert not set(leakage_columns).intersection(X_frame.columns)
assert y.isin([0, 1, 2, 3]).all()

print(f"Fitur sebelum encoding kota: {X_frame.shape[1]} kolom")
print("Kolom penyebab leakage dan kolom operasional tertunda yang dibuang:", leakage_columns)
print("Daftar fitur input sebelum encoding:")
print(X_frame.columns.tolist())
print(f"Dimensi X sebelum encoding: {X_frame.shape}; dimensi y: {y.shape}")
display(y.value_counts().sort_index().rename(index={0: "0 — Low", 1: "1 — Moderate", 2: "2 — High", 3: "3 — Extreme"}).to_frame("Sampel"))

Fitur sebelum encoding kota: 27 kolom
Kolom penyebab leakage dan kolom operasional tertunda yang dibuang: ['target_temp_max_besok', 'target_apparent_temp_besok', 'oni_anomaly', 'oni_sst_total', 'time', 'day_of_year']
Daftar fitur input sebelum encoding:
['radiation_sum', 'temp_max', 'apparent_temp_max', 'wind_speed_max', 'precip_sum', 'kota', 'latitude', 'longitude', 'elevation', 'oni_anom_lag1m', 'oni_anom_lag2m', 'oni_anom_lag3m', 'enso_phase', 'temp_max_lag1d', 'apparent_temp_lag1d', 'precip_lag1d', 'temp_max_lag2d', 'apparent_temp_lag2d', 'precip_lag2d', 'temp_max_lag3d', 'apparent_temp_lag3d', 'precip_lag3d', 'month', 'sin_doy', 'cos_doy', 'temp_range_or_heat_gap', 'apparent_temp_diff_1d']
Dimensi X sebelum encoding: (14952, 27); dimensi y: (14952,)


,Sampel
heat_risk_besok,
0 — Low,796
1 — Moderate,4618
2 — High,7813
3 — Extreme,1725


## Bab 5.3 — Chronological data splitting

In [4]:
train_mask = df_clean["time"] <= "2024-12-31"
val_mask = df_clean["time"].between("2025-01-01", "2025-12-31")
test_mask = df_clean["time"] >= "2026-01-01"

split_frames = {
    "Train": df_clean.loc[train_mask].reset_index(drop=True),
    "Validation": df_clean.loc[val_mask].reset_index(drop=True),
    "Test": df_clean.loc[test_mask].reset_index(drop=True),
}
split_y = {name: frame["heat_risk_besok"].astype("int64") for name, frame in split_frames.items()}
assert sum(len(frame) for frame in split_frames.values()) == len(df_clean)
assert split_frames["Train"]["time"].max() < split_frames["Validation"]["time"].min()
assert split_frames["Validation"]["time"].max() < split_frames["Test"]["time"].min()

split_rows = []
for name, frame in split_frames.items():
    split_rows.append({
        "Split": name,
        "Sampel": len(frame),
        "Persentase (%)": 100 * len(frame) / len(df_clean),
        "Tanggal awal": frame["time"].min().date(),
        "Tanggal akhir": frame["time"].max().date(),
    })
print("Ukuran dan rentang split:")
display(pd.DataFrame(split_rows).set_index("Split").round({"Persentase (%)": 2}))

distribution_rows = []
for name, labels in split_y.items():
    proportions = labels.value_counts(normalize=True).reindex([0, 1, 2, 3], fill_value=0) * 100
    distribution_rows.append({"Split": name, **{f"{label} — {class_name} (%)": proportions[label] for label, class_name in enumerate(["Low", "Moderate", "High", "Extreme"])}})
print("Proporsi kelas per split:")
display(pd.DataFrame(distribution_rows).set_index("Split").round(2))

Ukuran dan rentang split:


,Sampel,Persentase (%),Tanggal awal,Tanggal akhir
Split,,,,
Train,10206,68.26,2021-01-04,2024-12-31
Validation,2555,17.09,2025-01-01,2025-12-31
Test,2191,14.65,2026-01-01,2026-11-09


Proporsi kelas per split:


,0 — Low (%),1 — Moderate (%),2 — High (%),3 — Extreme (%)
Split,,,,
Train,5.73,31.62,51.23,11.41
Validation,4.58,29.35,55.77,10.29
Test,4.29,29.26,52.90,13.56


## Bab 5.4 — Data transformation tabular dan sequence LSTM

`kota` di-one-hot encode dengan kolom disejajarkan antarsplit. Hanya fitur numerik kontinyu yang distandardisasi; scaler di-fit pada train saja. Sliding windows tujuh hari dibuat terpisah di tiap kota dan split. Setiap label sequence adalah target `t+1` pada baris terakhir window.

In [5]:
base_features = [column for column in X_frame.columns if column != "kota"]
dummy_columns = sorted(f"kota_{city}" for city in df_clean["kota"].unique())

def encode_split(frame):
    numeric = frame[base_features].astype("float64").reset_index(drop=True)
    city_dummies = pd.get_dummies(frame["kota"], prefix="kota", dtype=int).reset_index(drop=True)
    encoded = pd.concat([numeric, city_dummies], axis=1)
    return encoded.reindex(columns=base_features + dummy_columns, fill_value=0)

X_train_raw = encode_split(split_frames["Train"])
X_val_raw = encode_split(split_frames["Validation"])
X_test_raw = encode_split(split_frames["Test"])
assert X_train_raw.columns.equals(X_val_raw.columns)
assert X_train_raw.columns.equals(X_test_raw.columns)

numeric_columns = base_features
scaler = StandardScaler()
X_train = X_train_raw.copy()
X_val = X_val_raw.copy()
X_test = X_test_raw.copy()
X_train.loc[:, numeric_columns] = scaler.fit_transform(X_train_raw[numeric_columns])
X_val.loc[:, numeric_columns] = scaler.transform(X_val_raw[numeric_columns])
X_test.loc[:, numeric_columns] = scaler.transform(X_test_raw[numeric_columns])

y_train = split_y["Train"].to_numpy(dtype=np.int64)
y_val = split_y["Validation"].to_numpy(dtype=np.int64)
y_test = split_y["Test"].to_numpy(dtype=np.int64)
print(f"Fitur model tabular setelah one-hot encoding: {X_train.shape[1]}")
print(f"Fitur kontinyu di-scale: {len(numeric_columns)}; fitur biner one-hot tidak di-scale.")
print(f"X_train: {X_train.shape}; X_val: {X_val.shape}; X_test: {X_test.shape}")
print("Nama fitur final:")
print(X_train.columns.tolist())

def make_city_windows(features, labels, metadata, timesteps=7):
    feature_array = features.to_numpy(dtype=np.float32)
    label_array = np.asarray(labels, dtype=np.int64)
    city_array = metadata["kota"].to_numpy()
    windows, window_labels = [], []
    for city in pd.unique(city_array):
        indices = np.flatnonzero(city_array == city)
        city_features = feature_array[indices]
        city_labels = label_array[indices]
        for end in range(timesteps - 1, len(indices)):
            windows.append(city_features[end - timesteps + 1:end + 1])
            window_labels.append(city_labels[end])
    return np.stack(windows), np.asarray(window_labels, dtype=np.int64)

sequence_metadata = {name: frame[["kota", "time"]].copy() for name, frame in split_frames.items()}
X_train_seq, y_train_seq = make_city_windows(X_train, y_train, sequence_metadata["Train"])
X_val_seq, y_val_seq = make_city_windows(X_val, y_val, sequence_metadata["Validation"])
X_test_seq, y_test_seq = make_city_windows(X_test, y_test, sequence_metadata["Test"])
print(f"X_train_seq: {X_train_seq.shape}; y_train_seq: {y_train_seq.shape}")
print(f"X_val_seq: {X_val_seq.shape}; y_val_seq: {y_val_seq.shape}")
print(f"X_test_seq: {X_test_seq.shape}; y_test_seq: {y_test_seq.shape}")

Fitur model tabular setelah one-hot encoding: 33
Fitur kontinyu di-scale: 26; fitur biner one-hot tidak di-scale.
X_train: (10206, 33); X_val: (2555, 33); X_test: (2191, 33)
Nama fitur final:
['radiation_sum', 'temp_max', 'apparent_temp_max', 'wind_speed_max', 'precip_sum', 'latitude', 'longitude', 'elevation', 'oni_anom_lag1m', 'oni_anom_lag2m', 'oni_anom_lag3m', 'enso_phase', 'temp_max_lag1d', 'apparent_temp_lag1d', 'precip_lag1d', 'temp_max_lag2d', 'apparent_temp_lag2d', 'precip_lag2d', 'temp_max_lag3d', 'apparent_temp_lag3d', 'precip_lag3d', 'month', 'sin_doy', 'cos_doy', 'temp_range_or_heat_gap', 'apparent_temp_diff_1d', 'kota_Balikpapan', 'kota_Jakarta', 'kota_Jayapura', 'kota_Madiun', 'kota_Makasar', 'kota_Medan', 'kota_Surabaya']
X_train_seq: (10164, 7, 33); y_train_seq: (10164,)
X_val_seq: (2513, 7, 33); y_val_seq: (2513,)
X_test_seq: (2149, 7, 33); y_test_seq: (2149,)


## Bab 5.5 — Data balancing (training only)

In [6]:
class_ids = np.unique(y_train)
class_weight_values = compute_class_weight(
    class_weight="balanced", classes=class_ids, y=y_train,
)
class_weights = {int(class_id): float(weight) for class_id, weight in zip(class_ids, class_weight_values)}
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)
class_names = {0: "Low", 1: "Moderate", 2: "High", 3: "Extreme"}
weights_table = pd.DataFrame([
    {"Kelas": class_id, "Nama": class_names[class_id],
     "Jumlah train": int(np.sum(y_train == class_id)), "Bobot": class_weights[class_id]}
    for class_id in class_ids
])
display(weights_table.round({"Bobot": 4}))
print(f"Bobot sampel train: {sample_weights.shape}; hanya berasal dari y_train.")
print("Class weighting dipilih alih-alih SMOTE agar urutan temporal dan pola meteorologis sintetis tidak terdistorsi.")

,Kelas,Nama,Jumlah train,Bobot
0,0,Low,585,4.3615
1,1,Moderate,3227,0.7907
2,2,High,5229,0.4880
3,3,Extreme,1165,2.1901


Bobot sampel train: (10206,); hanya berasal dari y_train.
Class weighting dipilih alih-alih SMOTE agar urutan temporal dan pola meteorologis sintetis tidak terdistorsi.


## Simpan artefak data siap latih

In [7]:
for name, features, labels in [
    ("train", X_train, y_train),
    ("val", X_val, y_val),
    ("test", X_test, y_test),
]:
    features.to_csv(OUTPUT_DIR / f"X_{name}.csv", index=False)
    pd.Series(labels, name="heat_risk_besok").to_csv(OUTPUT_DIR / f"y_{name}.csv", index=False)

np.savez_compressed(
    OUTPUT_DIR / "lstm_sequences.npz",
    X_train_seq=X_train_seq, y_train_seq=y_train_seq,
    X_val_seq=X_val_seq, y_val_seq=y_val_seq,
    X_test_seq=X_test_seq, y_test_seq=y_test_seq,
)
joblib.dump(scaler, OUTPUT_DIR / "scaler.joblib")
with open(OUTPUT_DIR / "class_weights.json", "w", encoding="utf-8") as file:
    json.dump(
        {class_names[class_id]: weight for class_id, weight in class_weights.items()},
        file, indent=2, ensure_ascii=False,
    )
print(f"Artefak disimpan ke {OUTPUT_DIR.resolve()}:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(f"- {path.name}: {path.stat().st_size:,} bytes")

Artefak disimpan ke D:\Lloyd\College\Semester5\DataScience\UAS\ProjectTermal\Termal-Guard\data\processed:
- class_weights.json: 131 bytes
- lstm_sequences.npz: 1,057,503 bytes
- scaler.joblib: 1,863 bytes
- X_test.csv: 1,150,260 bytes
- X_train.csv: 5,368,688 bytes
- X_val.csv: 1,353,206 bytes
- y_test.csv: 6,590 bytes
- y_train.csv: 30,635 bytes
- y_val.csv: 7,682 bytes


## Bab 5.6 — Ringkasan data preparation (Apa–Mengapa–Bagaimana)

| Tahap | Apa | Mengapa | Bagaimana |
|---|---|---|---|
| Cleaning | Hapus baris yang mengandung NaN; pertahankan outlier cuaca valid. | NaN berasal dari lag dan target pada batas deret, bukan imputasi observasi; outlier adalah kejadian fisis. | `dropna()` menghasilkan 14.952 baris, 2.136 per kota, tanpa duplikat. |
| Feature engineering/selection | Label empat kelas tropis; fitur siklikal dan heat-gap/perubahan suhu; buang target masa depan dan ONI bulan berjalan. | Menangkap musim dan kondisi termal tanpa memasukkan informasi masa depan/tertunda. | Cut 33/36/39°C (`right=False`), `sin_doy`, `cos_doy`, selisih suhu; sisakan fitur historis. |
| Split | Train 2021–2024, validation 2025, test 2026. | Menghormati sifat non-i.i.d. dan menahan test untuk evaluasi akhir. | Pemisahan tanggal sebelum scaling/encoding, tanpa shuffle. |
| Transformasi | One-hot kota, standardisasi numerik train-only, window sequence tujuh hari per kota dan split. | Menyamakan skala tanpa leakage dan menyediakan format untuk model tabular maupun LSTM. | 33 fitur hasil encoding; scaler fit hanya pada X_train. |
| Balancing | Class weights dan sample weights pada train saja. | Mengatasi ketimpangan tanpa merusak urutan deret waktu dengan sampel sintetis. | `compute_class_weight`/`compute_sample_weight` menggunakan `y_train`. |
| Artefak | Tabel train/val/test, sequence LSTM, scaler, dan bobot kelas. | Menyediakan input konsisten untuk tahap modeling dan reproduksibilitas. | Disimpan di `data/processed/`; dibuat ulang dengan menjalankan notebook dari atas ke bawah. |